# **Limpieza general de datos**
### **Carga de datos**

In [191]:
import pandas as pd

ruta_datos = '../data/data.csv'
df = pd.read_csv(ruta_datos)

### **Vista inicial del DataFrame**

In [178]:
print('Vista inicial del DataFrame:')
print('------------------------------------------')
print(df)

Vista inicial del DataFrame:
------------------------------------------
           country  location_id        observation_id               user_id  \
0      Afghanistan          160  edu_6447710491377664  edu_4771167334563840   
1      Afghanistan          160  edu_5214105756762112  edu_4525461102395392   
2      Afghanistan          160  edu_6358619313668096  edu_6260553241853952   
3      Afghanistan          160  edu_5023246000062464  edu_6732475616722944   
4      Afghanistan          160  edu_6525867185668096  edu_5539453241393152   
...            ...          ...                   ...                   ...   
23347        Yemen          157  edu_6642378533502976  edu_5383239220068352   
23348        Yemen          157  edu_6049497825411072  edu_5562530184560640   
23349        Yemen          157  edu_5184529160732672  edu_6481353718104064   
23350        Yemen          157  edu_6116795667972096  edu_5986846446452736   
23351        Yemen          157  edu_5354576713875456  edu_

### **Información inicial del DataFrame**

In [179]:
print('Información inicial del DataFrame:')
print('------------------------------------------')
df.info()

Información inicial del DataFrame:
------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23352 entries, 0 to 23351
Columns: 122 entries, country to edu_ch4_help_99
dtypes: float64(93), int64(11), object(18)
memory usage: 21.7+ MB


### **Conversión de `submission_time` a tipo `datetime`.**

In [180]:
df['submission_time'] = pd.to_datetime(df['submission_time'], errors = 'coerce')

# Verificación del cambio
print('Columna submission_time transformada')
print('------------------------------------------')
print(df['submission_time'])

Columna submission_time transformada
------------------------------------------
0       2021-05-28 19:53:45.487000+00:00
1       2021-05-28 20:22:11.486000+00:00
2       2021-05-28 21:05:35.887000+00:00
3       2021-05-28 21:28:21.402000+00:00
4       2021-05-28 23:25:45.569000+00:00
                      ...               
23347   2021-06-19 23:59:32.615000+00:00
23348   2021-06-20 02:31:03.343000+00:00
23349   2021-06-19 23:25:49.240000+00:00
23350   2021-06-20 02:55:07.400000+00:00
23351   2021-06-19 23:50:11.885000+00:00
Name: submission_time, Length: 23352, dtype: datetime64[ns, UTC]


La variable `ethnicity` será eliminada debido a su formato inconsistente (múltiples selecciones concatenadas) y porque no aporta información relevante para los objetivos de segmentación, predicción o análisis de acceso educativo.

In [181]:
df.drop(columns = 'ethnicity', inplace = True)
print("Nuevas dimensiones del DataFrame:", df.shape)

Nuevas dimensiones del DataFrame: (23352, 121)


Borramos las columnas de opción múltiple ya que el dataset contiene la misma información pero separada por nuevas columnas.

In [192]:
columnas_multiple = [col for col in df.columns if df[col].astype(str).str.contains('\^', regex=True).any()]
df.drop(columns = columnas_multiple, inplace = True)
print('Nuevas dimensiones del DataFrame:', df.shape)

<>:1: SyntaxWarning: invalid escape sequence '\^'
<>:1: SyntaxWarning: invalid escape sequence '\^'
C:\Users\andy-\AppData\Local\Temp\ipykernel_10208\2687228767.py:1: SyntaxWarning: invalid escape sequence '\^'
  columnas_multiple = [col for col in df.columns if df[col].astype(str).str.contains('\^', regex=True).any()]


Nuevas dimensiones del DataFrame: (23352, 113)


### **Tratamiento de valores nulos**

In [193]:
# Cantidad y porcentaje de valores nulos
valores_nulos = df.isnull().sum()
porcentaje_nulos = (valores_nulos / len(df)) * 100

nulos_df = pd.DataFrame({
    'Valores_Nulos': valores_nulos,
    'Porcentaje_Nulos': porcentaje_nulos
}).sort_values(by = 'Porcentaje_Nulos', ascending = False)

print('Variables con más del 90% de valores nulos:')
nulos_mayor_90 = nulos_df[nulos_df['Porcentaje_Nulos'] > 90].index.tolist()
for i, col in enumerate(nulos_mayor_90):
    print(col, end = '\t')
    if (i + 1) % 2 == 0:
        print()
print(f'\nTotal: {len(nulos_mayor_90)}')

Variables con más del 90% de valores nulos:
edu_ch4_school_reopen	edu_ch3_school_reopen	
edu_ch4_no_school_why	edu_ch2_school_reopen	
edu_ch3_no_school_why	edu_ch1_school_reopen	
edu_ch2_no_school_why	edu_ch4_internet_access	
edu_ch1_no_school_why	edu_ch4_help_3	
edu_ch4_help_1	edu_ch4_help_4	
edu_ch4_help_5	edu_ch4_help_6	
edu_ch4_help_2	edu_ch4_help_0	
edu_ch4_help_99	edu_ch4_remote_method_2	
edu_ch4_remote_method_4	edu_ch4_remote_method_3	
edu_ch4_remote_method_5	edu_ch4_remote_method_1	
edu_ch4_support_2	edu_ch4_support_0	
edu_ch4_support_1	edu_ch4_support	

Total: 26


Eliminamos las variables con más del 90% de valores nulos porque aportan muy poca información útil y no son relevantes para los objetivos del proyecto (segmentación, predicción y análisis de acceso educativo).

In [146]:
df.drop(columns = nulos_mayor_90, inplace = True)
print("Nuevas dimensiones del DataFrame:", df.shape)

Nuevas dimensiones del DataFrame: (23352, 94)


In [175]:
columnas_con_ch1 = [col for col in df.columns.tolist() if 'ch1' in col]
columnas_sin_edu = [col for col in df.columns.tolist() if 'edu' not in col]
columnas_sin_edu = [col for col in columnas_sin_edu if 'id' not in col]
columnas_sin_edu.remove('submission_time')
variables_importantes = columnas_con_ch1 + columnas_sin_edu

valores_nulos = df[variables_importantes].isnull().sum()
porcentaje_nulos = (valores_nulos / len(df)) * 100
nulos_df = pd.DataFrame({
    'Valores_Nulos': valores_nulos,
    'Porcentaje_Nulos': porcentaje_nulos
}).sort_values(by = 'Porcentaje_Nulos', ascending = False)
print(nulos_df[:25])
print(len(nulos_df))

                         Valores_Nulos  Porcentaje_Nulos
edu_ch1_internet_access          13347         57.155704
edu_ch1_help                     12231         52.376670
edu_ch1_help_6                   12231         52.376670
edu_ch1_help_2                   12231         52.376670
edu_ch1_help_5                   12231         52.376670
edu_ch1_help_4                   12231         52.376670
edu_ch1_help_1                   12231         52.376670
edu_ch1_help_3                   12231         52.376670
edu_ch1_help_99                  12231         52.376670
edu_ch1_help_0                   12231         52.376670
edu_ch1_remote_method            12222         52.338129
edu_ch1_remote_method_4          12222         52.338129
edu_ch1_remote_method_1          12222         52.338129
edu_ch1_remote_method_2          12222         52.338129
edu_ch1_remote_method_3          12222         52.338129
edu_ch1_remote_method_5          12222         52.338129
edu_ch1_support                

# **Rayos!!!**
Esta forma de limpiar está complicada. Muchas columnas innecesarias, muchas columnas necesarias, mucho trabajo :c Intentaré reorganizar el dataset tal que cada fila corresponda a un niño diferente (y no a un cuidador diferente). Nos vemos en la siguiente parte.